In [25]:
# all important imports 
using JLD2, DataFrames, CSV, StaticArrays, SciMLSensitivity, Lux, LinearAlgebra, DifferentialEquations, Plots,Random

In [26]:

true_data = data_loaded = CSV.read("C:/Ude_HH_model/ude_models/noisy_data.csv", DataFrame)
y = Float32.(true_data[!, "n"])
t_steps = Float32.(true_data[!, "Time"])



3001-element Vector{Float32}:
  0.0
  0.01
  0.02
  0.03
  0.04
  0.05
  0.06
  0.07
  0.08
  0.09
  0.1
  0.11
  0.12
  ⋮
 29.89
 29.9
 29.91
 29.92
 29.93
 29.94
 29.95
 29.96
 29.97
 29.98
 29.99
 30.0

In [27]:
rng = Random.seed!(42)
nn = Chain(Dense(1 => 32),
    Dense(32 => 32, tanh),
    Dense(32 => 1, sigmoid))
ps, st = Lux.setup(rng, nn)



((layer_1 = (weight = Float32[-0.15767582; -1.1307708; … ; 0.19089252; 0.42486224;;], bias = Float32[0.6402823, -0.15548313, 0.3585751, 0.47565484, -0.83354366, -0.7983309, -0.9632733, -0.35786855, 0.1624806, 0.93903005  …  0.86745214, 0.8148211, 0.38635254, -0.62182, -0.18979192, -0.6013012, -0.8030517, 0.56790996, -0.28980362, 0.96977556]), layer_2 = (weight = Float32[0.37709013 -0.4490932 … -0.3449148 -0.36564463; 0.054253716 0.49733526 … 0.35948935 0.24132304; … ; 0.49617618 0.08238792 … 0.18305631 -0.1491749; -0.021193195 -0.16762535 … -0.43130484 0.3783914], bias = Float32[0.03878094, -0.007602322, -0.11970364, -0.098564304, -0.025683658, -0.039688274, -0.12524073, 0.012478712, -0.054133095, 0.12872824  …  0.040868536, 0.09221246, -0.08388274, 0.08786137, -0.1030199, -0.08795758, 0.07973033, 0.059021942, 0.066383034, 0.067760795]), layer_3 = (weight = Float32[-0.29468712 -0.103986636 … -0.19075362 -0.13402101], bias = Float32[0.10804534])), (layer_1 = NamedTuple(), layer_2 = Name

In [28]:
# 1. Load synthetic data and time steps
true_data = CSV.read("C:/Ude_HH_model/ude_models/synthetic_data.csv", DataFrame)
t_steps = Float32.(true_data[!, "Time"])
ture_n = Float32.(true_data[!, "n"])

3001-element Vector{Float32}:
 0.317
 0.31700262
 0.317008
 0.3170161
 0.31702694
 0.3170405
 0.31705678
 0.31707576
 0.3170974
 0.31712174
 0.31714872
 0.31717837
 0.31721067
 ⋮
 0.40159258
 0.4017208
 0.40184993
 0.40198
 0.40211102
 0.40224305
 0.402376
 0.40250996
 0.40264493
 0.4027809
 0.40291786
 0.40305588

In [29]:
# 2. Hodgkin-Huxley Model Parameters

const g_na = 120.0f0
const g_k = 36.0f0
const g_l = 0.3f0
const c_m = 1.0f0
const I_ext = 10.0f0
const E_na = 50.0f0
const E_k = -77.0f0
const E_l = -54.4f0

-54.4f0

In [30]:
# 3. Rate Functions for Gating Variables
alpha_n(V) = abs(V + 55.0f0) < 1.0f-6 ? 0.1f0 : 0.01f0 * (V + 55.0f0) / (1.0f0 - exp(-(V + 55.0f0) / 10.0f0))
beta_n(V) = 0.125f0 * exp(-(V + 65.0f0) / 80.0f0)

alpha_m(V) = abs(V + 40.0f0) < 1.0f-6 ? 1.0f0 : 0.1f0 * (V + 40.0f0) / (1.0f0 - exp(-(V + 40.0f0) / 10.0f0))
beta_m(V) = 4.0f0 * exp(-(V + 65.0f0) / 18.0f0)

alpha_h(V) = 0.07f0 * exp(-(V + 65.0f0) / 20.0f0)
beta_h(V) = 1.0f0 / (1.0f0 + exp(-(V + 35.0f0) / 10.0f0))

beta_h (generic function with 1 method)

In [31]:
# 4. Neural Network Architecture & State Setup
nn = Chain(
    Dense(1 => 32),
    Dense(32 => 32, tanh),
    Dense(32 => 1, sigmoid)
)
rng = Random.seed!(42)
_, st = Lux.setup(rng, nn)


((layer_1 = (weight = Float32[-0.15767582; -1.1307708; … ; 0.19089252; 0.42486224;;], bias = Float32[0.6402823, -0.15548313, 0.3585751, 0.47565484, -0.83354366, -0.7983309, -0.9632733, -0.35786855, 0.1624806, 0.93903005  …  0.86745214, 0.8148211, 0.38635254, -0.62182, -0.18979192, -0.6013012, -0.8030517, 0.56790996, -0.28980362, 0.96977556]), layer_2 = (weight = Float32[0.37709013 -0.4490932 … -0.3449148 -0.36564463; 0.054253716 0.49733526 … 0.35948935 0.24132304; … ; 0.49617618 0.08238792 … 0.18305631 -0.1491749; -0.021193195 -0.16762535 … -0.43130484 0.3783914], bias = Float32[0.03878094, -0.007602322, -0.11970364, -0.098564304, -0.025683658, -0.039688274, -0.12524073, 0.012478712, -0.054133095, 0.12872824  …  0.040868536, 0.09221246, -0.08388274, 0.08786137, -0.1030199, -0.08795758, 0.07973033, 0.059021942, 0.066383034, 0.067760795]), layer_3 = (weight = Float32[-0.29468712 -0.103986636 … -0.19075362 -0.13402101], bias = Float32[0.10804534])), (layer_1 = NamedTuple(), layer_2 = Name

In [32]:
# 5. Load Trained Parameters
ps = load("C:/Ude_HH_model/ude_models/leaning_parameter2.jld2", "p")


┌ Warning: type NamedTuple{(:weight, :bias),Tuple{ComponentArrays.ViewAxis{1:32,nothing,ComponentArrays.ShapedAxis{(32, 1)}},ComponentArrays.ViewAxis{33:64,nothing,ComponentArrays.Shaped1DAxis{(32,)}}}} does not exist in workspace; reconstructing
└ @ JLD2 C:\Users\ADMIN\.julia\packages\JLD2\ADycq\src\data\reconstructing_datatypes.jl:472
┌ Warning: type NamedTuple{(:weight, :bias),Tuple{ComponentArrays.ViewAxis{1:1024,nothing,ComponentArrays.ShapedAxis{(32, 32)}},ComponentArrays.ViewAxis{1025:1056,nothing,ComponentArrays.Shaped1DAxis{(32,)}}}} does not exist in workspace; reconstructing
└ @ JLD2 C:\Users\ADMIN\.julia\packages\JLD2\ADycq\src\data\reconstructing_datatypes.jl:472
┌ Warning: type NamedTuple{(:weight, :bias),Tuple{ComponentArrays.ViewAxis{1:32,nothing,ComponentArrays.ShapedAxis{(1, 32)}},ComponentArrays.ViewAxis{33:33,nothing,ComponentArrays.Shaped1DAxis{(1,)}}}} does not exist in workspace; reconstructing
└ @ JLD2 C:\Users\ADMIN\.julia\packages\JLD2\ADycq\src\data\reconstru

Reconstruct@ComponentArray{Float32,1,Vector{Float32},Tuple{Axis{Reconstruct@NamedTuple{(:layer_1, :layer_2, :layer_3),Tuple{ViewAxis{1:64,Reconstruct@NamedTuple{(:weight, :bias),Tuple{ViewAxis{1:32,nothing,ShapedAxis{(32, 1)}},ViewAxis{33:64,nothing,Shaped1DAxis{(32,)}}}}(),Axis{Reconstruct@NamedTuple{(:weight, :bias),Tuple{ViewAxis{1:32,nothing,ShapedAxis{(32, 1)}},ViewAxis{33:64,nothing,Shaped1DAxis{(32,)}}}}()}},ViewAxis{65:1120,Reconstruct@NamedTuple{(:weight, :bias),Tuple{ViewAxis{1:1024,nothing,ShapedAxis{(32, 32)}},ViewAxis{1025:1056,nothing,Shaped1DAxis{(32,)}}}}(),Axis{Reconstruct@NamedTuple{(:weight, :bias),Tuple{ViewAxis{1:1024,nothing,ShapedAxis{(32, 32)}},ViewAxis{1025:1056,nothing,Shaped1DAxis{(32,)}}}}()}},ViewAxis{1121:1153,Reconstruct@NamedTuple{(:weight, :bias),Tuple{ViewAxis{1:32,nothing,ShapedAxis{(1, 32)}},ViewAxis{33:33,nothing,Shaped1DAxis{(1,)}}}}(),Axis{Reconstruct@NamedTuple{(:weight, :bias),Tuple{ViewAxis{1:32,nothing,ShapedAxis{(1, 32)}},ViewAxis{33:33,nothi

In [33]:
# 6. Universal Differential Equations System

function ude_hh!(du, u, p, t)
    V, m, h, n = u
    ps = p

    pred_n, _ = nn(@SVector[n], ps, st)

    du[1] = 1 / c_m * (I_ext - g_na * m^3 * h * (V - E_na) - g_k * pred_n[1] * (V - E_k) - g_l * (V - E_l))
    du[2] = alpha_m(V) * (1 - m) - beta_m(V) * (m)
    du[3] = alpha_h(V) * (1 - h) - beta_h(V) * (h)
    du[4] = alpha_n(V) * (1 - n) - beta_n(V) * (n)
end

# 7. Initial Conditions & ODE Solution
const u_0 = [-65.0f0, 0.05f0, 0.6f0, 0.317f0]
tspan = (0.0f0, 30.0f0)


(0.0f0, 30.0f0)

In [34]:
prob = ODEProblem(ude_hh!, u_0, tspan, ps)
sol = solve(prob, Rosenbrock23(), reltol=1e-6, abstol=1e-6, saveat=t_steps, sensealg=ForwardDiffSensitivity())


LoadError: ArgumentError: field layer_1 not found

In [ ]:
sol.t
n = sol[4, :]
y = Float64[nn(@SVector[n[j]], ps, st)[1][1] for j in 1:length(n)]


In [ ]:
# 9. Diverse Candidate Basis Functions Library (28 Functions)
basis_func = [
    n -> 1.0,
    n -> n,
    n -> n^2,
    n -> n^3,
    n -> n^4,
    n -> n^5,
    n -> n^6,
    n -> n^7,
    n -> n^8,
    n -> sqrt(max(0.0, n)),
    n -> sin(n),
    n -> cos(n),
    n -> sin(2 * n),
    n -> cos(2 * n),
    n -> sin(3 * n),
    n -> cos(3 * n),
    n -> sin(pi * n),
    n -> cos(pi * n),
    n -> exp(n),
    n -> exp(-n),
    n -> exp(2 * n),
    n -> exp(-2 * n),
    n -> sinh(n),
    n -> cosh(n),
    n -> tanh(n),
    n -> log(1.0 + max(0.0, n)),
    n -> 1.0 / (1.0 + n),
    n -> n / (1.0 + n)
]

In [ ]:
basis_names = [
    "1", "n", "n^2", "n^3", "n^4", "n^5", "n^6", "n^7", "n^8",
    "sqrt(n)",
    "sin(n)", "cos(n)", "sin(2n)", "cos(2n)", "sin(3n)", "cos(3n)", "sin(πn)", "cos(πn)",
    "exp(n)", "exp(-n)", "exp(2n)", "exp(-2n)", "sinh(n)", "cosh(n)", "tanh(n)",
    "log(1+n)", "1/(1+n)", "n/(1+n)"
]

In [ ]:
# 10. Construct Design Matrix (ϕ)
basis_l = length(basis_func)
point_l = length(n)
ϕ = [basis_func[i](n[j]) for j in 1:point_l, i in 1:basis_l]
y = Float64[nn(@SVector[n[j]], ps, st)[1][1] for j in 1:length(n)]
num_basis = size(ϕ, 2)
β = ϕ \ y